In [4]:
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [5]:
def create_mfcc_features(df, clean_dir, n_mfcc=13, hop_length=512, sr=16000):
    clean_dir = Path(clean_dir)
    mfcc_features_list = []
    
    for f in tqdm(df.fname):
        y, _ = librosa.load(clean_dir / f, sr=sr)
        n_fft = min(2048, len(y))
        mfcc = librosa.feature.mfcc(y=y, 
                                    sr=sr, 
                                    n_mfcc=n_mfcc, 
                                    n_fft=n_fft, 
                                    hop_length=hop_length)
        mfcc_mean = np.mean(mfcc, axis=1)
        mfcc_std = np.std(mfcc, axis=1)
        mfcc_features = np.concatenate([mfcc_mean, mfcc_std])
        mfcc_features_list.append(mfcc_features)

    X = np.array(mfcc_features_list)
    y = df['label'].values

    return X, y

Load train/test DataFrame

In [6]:
train_csv_path = 'meta/train_post_competition.csv'
train_df = pd.read_csv(train_csv_path)

test_csv_path = 'meta/test_post_competition_scoring_clips.csv'
test_df = pd.read_csv(test_csv_path)

In [7]:
# musical_instruments = [
#     'Hi-hat', 'Saxophone', 'Trumpet', 'Glockenspiel', 'Cello', 'Knock',
#     'Gunshot_or_gunfire', 'Clarinet', 'Computer_keyboard',
#     'Keys_jangling', 'Snare_drum', 'Writing', 'Laughter', 'Tearing',
#     'Fart', 'Oboe', 'Flute', 'Cough', 'Telephone', 'Bark', 'Chime',
#     'Bass_drum', 'Bus', 'Squeak', 'Scissors', 'Harmonica', 'Gong',
#     'Microwave_oven', 'Burping_or_eructation', 'Double_bass',
#     'Shatter', 'Fireworks', 'Tambourine', 'Cowbell', 'Electric_piano',
#     'Meow', 'Drawer_open_or_close', 'Applause', 'Acoustic_guitar',
#     'Violin_or_fiddle', 'Finger_snapping'
# ]

musical_instruments = [
    'Hi-hat', 'Saxophone', 'Trumpet', 'Glockenspiel', 'Cello', 
    'Clarinet', 'Oboe', 'Flute', 'Bass_drum', 'Double_bass',
    'Tambourine', 'Cowbell', 'Electric_piano', 'Harmonica', 
    'Acoustic_guitar', 'Violin_or_fiddle'
]

train_df = train_df[train_df['label'].isin(musical_instruments)]
test_df = test_df[test_df['label'].isin(musical_instruments)]

In [8]:
train_df = train_df[train_df['fname'].apply(lambda x: os.path.exists(os.path.join('train', x)))]
train_df = train_df[train_df['manually_verified'] == 1]
train_df = train_df.reset_index(drop=True)

test_df = test_df[test_df['fname'].apply(lambda x: os.path.exists(os.path.join('test', x)))]
test_df = test_df.reset_index(drop=True)

Load data

In [9]:
X_train, y_train = create_mfcc_features(train_df, 'clean_train')
X_test, y_test = create_mfcc_features(test_df, 'clean_test')

100%|██████████| 790/790 [02:43<00:00,  4.83it/s]


Train SVM

In [10]:
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

SVC(C=1.0, break_ties=False, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma='scale', kernel='linear',
    max_iter=-1, probability=False, random_state=None, shrinking=True,
    tol=0.001, verbose=False)

Evaluate SVM

In [11]:
y_pred = svm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print('Accuracy:', accuracy)
print('\nClassification Report:\n', class_report)

output_file = 'svm_evaluation.txt'

with open(output_file, 'w') as file:
    file.write(f'Accuracy: {accuracy}\n')
    file.write('\nClassification Report:\n')
    file.write(class_report)

Accuracy: 0.7341772151898734

Classification Report:
                   precision    recall  f1-score   support

 Acoustic_guitar       0.52      0.71      0.60        45
       Bass_drum       0.79      0.96      0.87        28
           Cello       0.76      0.78      0.77        54
        Clarinet       0.62      0.71      0.67        56
         Cowbell       0.79      0.90      0.84        42
     Double_bass       0.86      0.75      0.80        40
  Electric_piano       0.68      0.78      0.72        32
           Flute       0.61      0.51      0.55        55
    Glockenspiel       0.90      0.62      0.73        29
       Harmonica       0.44      0.21      0.29        33
          Hi-hat       0.71      0.77      0.74        39
            Oboe       0.79      0.81      0.80        42
       Saxophone       0.73      0.74      0.73       110
      Tambourine       0.95      0.90      0.92        40
         Trumpet       0.79      0.62      0.70        37
Violin_or_fiddle 

Save SVM

In [12]:
model_filepath = 'svm_model.pkl'

with open(model_filepath, 'wb') as model_file:
    pickle.dump(svm, model_file)

print(f'SVM saved to {os.path.abspath(output_file)}')

SVM saved to c:\Users\grzes\Desktop\praca-inzynierska\svm_evaluation.txt
